# Pipeline Bronze para Silver — CineData Analytics

Este notebook implementa o pipeline de conformação, enriquecimento e higienização dos dados da camada **Bronze** para a camada **Silver**.

### Diretrizes de Engenharia e Padrões Aplicados:
- **Contratos Canônicos de Esquema**: Validação estrita de tipos e estrutura via `StructType` para cada tabela de destino.
- **Tradução e Nomenclatura**: Padronização dos nomes de campos em português (`snake_case`).
- **Rastreabilidade**: Preservação da linhagem temporal mantendo o `ingestion_datetime` gerado na ingestão Bronze.
- **Modularidade e Coesão**: Funções utilitárias puras e constantes de domínio aplicadas diretamente em cada etapa.

In [1]:
import os
import sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

from pyspark.sql import Column, DataFrame, Row, SparkSession, Window
from pyspark.sql.functions import (
    coalesce,
    col,
    create_map,
    explode,
    initcap,
    last,
    length,
    lit,
    regexp_extract,
    regexp_replace,
    row_number,
    split,
    to_date,
    trim,
    try_to_date,
    upper,
    when,
    year,
)
from pyspark.sql.functions import max as spark_max
from pyspark.sql.functions import min as spark_min
from pyspark.sql.types import (
    DateType,
    DecimalType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# Detecção dinâmica de ambiente: Databricks vs. Local
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if not IS_DATABRICKS:
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Resolução de caminhos do Lakehouse
WORKSPACE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = WORKSPACE_DIR / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"

# Inicialização ou obtenção da Sessão Spark
if IS_DATABRICKS:
    spark = SparkSession.builder.getOrCreate()
else:
    spark = (
        SparkSession
        .builder
        .appName("CineData_02_Bronze_to_Silver")
        .config("spark.driver.memory", "4g")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.sql.ansi.enabled", "false")
        .getOrCreate()
    )
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.ansi.enabled", "false")

# Provisionamento do banco de dados (database) da camada Silver
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# Função helper unificada para exibição rica no Databricks e compatível localmente
def display_dataframe(dataframe: DataFrame, row_limit: int = 10) -> None:
    """
    Exibe o DataFrame utilizando display() nativo no Databricks ou show() em ambiente local.
    """
    display_function = globals().get("display") or getattr(__builtins__, "display", None)
    if callable(display_function):
        display_function(dataframe)
    else:
        dataframe.show(row_limit, truncate=False)

def read_lakehouse_table(table_name: str, fallback_path: Path) -> DataFrame:
    """
    Lê a tabela gerenciada do catálogo no Databricks ou o arquivo físico em ambiente local.
    """
    if IS_DATABRICKS:
        return spark.table(table_name)
    return spark.read.parquet(str(fallback_path))

# Sentinelas de valores nulos ou ausentes na base bruta
DEFAULT_SENTINEL_VALUES = [
    "UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "NONE", "N/A", "NULL", ""
]

def enforce_dataframe_schema(
    dataframe: DataFrame,
    expected_schema: StructType,
    strict_columns: bool = True
) -> DataFrame:
    """
    Valida a presença de todos os campos definidos no StructType, aplica conversão defensiva
    e reordena as colunas. Em modo estrito, impede colunas não declaradas no contrato.
    """
    actual_column_names = set(dataframe.columns)
    expected_column_names = [field.name for field in expected_schema.fields]
    missing_columns = set(expected_column_names) - actual_column_names

    if missing_columns:
        raise ValueError(f"Colunas obrigatórias ausentes no DataFrame: {missing_columns}")

    if strict_columns:
        unexpected_columns = actual_column_names - set(expected_column_names)
        if unexpected_columns:
            raise ValueError(f"Colunas imprevistas encontradas no DataFrame: {unexpected_columns}")

    ordered_column_expressions = [
        col(field.name).cast(field.dataType).alias(field.name)
        for field in expected_schema.fields
    ]
    return dataframe.select(ordered_column_expressions)

def write_dataframe(
    dataframe: DataFrame,
    target_path: Path,
    table_name: str | None = None,
    storage_format: str = "delta" if IS_DATABRICKS else "parquet",
    save_mode: str = "overwrite"
) -> None:
    """
    Persiste o DataFrame no formato especificado preservando o schema existente.
    """
    writer = (
        dataframe.write
        .format(storage_format)
        .mode(save_mode)
    )
    if IS_DATABRICKS and table_name:
        writer.saveAsTable(table_name)
    else:
        writer.save(str(target_path))

def deduplicate_latest(
    dataframe: DataFrame,
    business_key_columns: list[str] | str | None = None,
    business_key_column: str | None = None,
    timestamp_column: str = "ingestion_datetime"
) -> DataFrame:
    """
    Garante a unicidade dos registros pela chave de negócio, mantendo exclusivamente
    a versão mais recente com base no timestamp de ingestão.
    """
    target_key = business_key_column if business_key_column is not None else business_key_columns
    if target_key is None:
        raise ValueError("Chave de negócio obrigatória para deduplicação.")
    key_columns_list = [target_key] if isinstance(target_key, str) else target_key
    window_by_business_key = (
        Window
        .partitionBy(*key_columns_list)
        .orderBy(col(timestamp_column).desc())
    )
    return (
        dataframe
        .withColumn("_row_order_number", row_number().over(window_by_business_key))
        .filter(col("_row_order_number") == 1)
        .drop("_row_order_number")
    )

def sanitize_string(column: Column, sentinel_values: list[str] | None = None) -> Column:
    """
    Higieniza colunas textuais convertendo valores sentinela e vazios para NULL.
    """
    active_sentinels = sentinel_values if sentinel_values is not None else DEFAULT_SENTINEL_VALUES
    uppercase_sentinels = [single_sentinel.upper() for single_sentinel in active_sentinels]
    trimmed_column = trim(column)
    return when(upper(trimmed_column).isin(uppercase_sentinels), lit(None)).otherwise(trimmed_column)

# Framework de Data Quality local para auditoria das transformações Silver
dq_execution_log: list[Row] = []

def execute_data_quality_check(
    table_name: str,
    check_name: str,
    target_dataframe: DataFrame,
    valid_condition: Column
) -> bool:
    """
    Avalia uma regra de integridade de dados, registrando o volume de registros
    em conformidade e violados no acumulador de auditoria.
    """
    total_records = target_dataframe.count()
    failed_records = target_dataframe.filter(~valid_condition).count()
    check_passed = (failed_records == 0)

    dq_execution_log.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_records=total_records,
            failed_records=failed_records,
            passed=check_passed,
            checked_at=datetime.now(ZoneInfo('America/Recife'))
        )
    )
    status_tag = "PASS" if check_passed else "FAIL"
    print(f"[{status_tag}] {table_name} | {check_name}: {failed_records}/{total_records} falhas.")
    return check_passed

def execute_uniqueness_check(
    table_name: str,
    check_name: str,
    target_dataframe: DataFrame,
    key_columns: list[str]
) -> bool:
    """
    Valida a unicidade da chave primária composta ou simples.
    """
    total_records = target_dataframe.count()
    duplicate_keys = (
        target_dataframe
        .groupBy(*key_columns)
        .count()
        .filter(col("count") > 1)
        .count()
    )
    check_passed = (duplicate_keys == 0)

    dq_execution_log.append(
        Row(
            table_name=table_name,
            check_name=check_name,
            total_records=total_records,
            failed_records=duplicate_keys,
            passed=check_passed,
            checked_at=datetime.now(ZoneInfo('America/Recife'))
        )
    )
    status_tag = "PASS" if check_passed else "FAIL"
    print(f"[{status_tag}] {table_name} | {check_name}: {duplicate_keys} duplicatas em {total_records} registros.")
    return check_passed

def persist_data_quality_log(target_directory: Path = SILVER_DIR) -> None:
    """
    Persiste o histórico acumulado de qualidade de dados na tabela de auditoria.
    """
    if dq_execution_log:
        dq_dataframe = spark.createDataFrame(dq_execution_log)
        write_dataframe(
            dataframe=dq_dataframe,
            target_path=target_directory / "tb_data_quality_log",
            table_name="silver.tb_data_quality_log" if IS_DATABRICKS else None,
            storage_format="delta" if IS_DATABRICKS else "parquet",
            save_mode="append"
        )
        print(f"Log de Data Quality persistido com sucesso em: {target_directory / 'tb_data_quality_log'}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/19 23:52:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/09/19 23:52:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/19 23:52:55 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/19 23:52:55 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


## 1. silver.tb_cotacao_dolar (Origem: bronze.tb_cotacao_dolar)

### Regras de Negócio:
- **Série Temporal Contínua**: Construção de calendário contínuo sem lacunas.
- **Forward Fill**: Dias sem cotação (finais de semana e feriados) herdam a última cotação útil disponível.

In [2]:
EXCHANGE_RATE_PRECISION = "decimal(18,4)"

TbCotacaoDolarSilverSchema = StructType([
    StructField("data_cotacao", DateType(), nullable=False),
    StructField("cotacao_compra", DecimalType(18, 4), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_tb_cotacao_dolar(raw_cotacao_dataframe: DataFrame) -> DataFrame:
    # Deduplicação diária: seleciona a cotação e ingestão mais recentes caso haja múltiplos registros no mesmo dia
    window_daily_quote = Window.partitionBy(to_date(col("dataHoraCotacao"))).orderBy(col("dataHoraCotacao").desc(), col("ingestion_datetime").desc())
    daily_quotes_dataframe = (
        raw_cotacao_dataframe
        .withColumn("data_cotacao", to_date(col("dataHoraCotacao")))
        .withColumn("_row_order_number", row_number().over(window_daily_quote))
        .filter(col("_row_order_number") == 1)
        .select(
            col("data_cotacao"),
            col("cotacaoCompra").cast(EXCHANGE_RATE_PRECISION).alias("cotacao_compra_bruta"),
            col("ingestion_datetime")
        )
    )

    # Como a API do Banco Central não possui cotações em finais de semana e feriados,
    # estruture o histórico de forma a garantir uma série temporal contínua,
    # gerando um calendário completo de datas entre a cotação mínima e máxima observadas.
    date_bounds = daily_quotes_dataframe.select(spark_min("data_cotacao"), spark_max("data_cotacao")).first()
    minimum_date, maximum_date = date_bounds[0], date_bounds[1]

    continuous_calendar = spark.sql(
        f"SELECT explode(sequence(to_date('{minimum_date}'), to_date('{maximum_date}'), interval 1 day)) as data_cotacao"
    )

    # Aplique uma técnica de preenchimento (Forward Fill) de modo que dias sem cotação
    # (finais de semana e feriados) recebam o valor do último dia útil disponível.
    forward_fill_window = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

    transformed_dataframe = (
        continuous_calendar
        .join(daily_quotes_dataframe, on="data_cotacao", how="left")
        .withColumn("cotacao_compra", last("cotacao_compra_bruta", ignorenulls=True).over(forward_fill_window))
        .withColumn("ingestion_datetime", last("ingestion_datetime", ignorenulls=True).over(forward_fill_window))
        .select("data_cotacao", "cotacao_compra", "ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbCotacaoDolarSilverSchema)

dataframe_cotacao_dolar_bronze = read_lakehouse_table("bronze.tb_cotacao_dolar", BRONZE_DIR / "bronze.tb_cotacao_dolar")
dataframe_cotacao_dolar_silver = transform_tb_cotacao_dolar(dataframe_cotacao_dolar_bronze)
write_dataframe(dataframe_cotacao_dolar_silver, SILVER_DIR / "silver.tb_cotacao_dolar", table_name="silver.tb_cotacao_dolar")
dataframe_cotacao_dolar_silver.printSchema()
display_dataframe(dataframe_cotacao_dolar_silver, 10)

execute_uniqueness_check("silver.tb_cotacao_dolar", "unicidade_data_cotacao", dataframe_cotacao_dolar_silver, ["data_cotacao"])
execute_data_quality_check("silver.tb_cotacao_dolar", "cotacao_compra_positiva", dataframe_cotacao_dolar_silver, col("cotacao_compra") > 0)


root
 |-- data_cotacao: date (nullable = false)
 |-- cotacao_compra: decimal(18,4) (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



DataFrame[data_cotacao: date, cotacao_compra: decimal(18,4), ingestion_datetime: timestamp]

[PASS] silver.tb_cotacao_dolar | unicidade_data_cotacao: 0 duplicatas em 5 registros.


[PASS] silver.tb_cotacao_dolar | cotacao_compra_positiva: 0/5 falhas.


True

## 2. silver.tb_info_filmes (Origem: bronze.tb_movies_info)

### Regras de Negócio:
- **Deduplicação**: Unicidade por `id` mantendo o registro mais recente por `ingestion_datetime`.
- **Tradução e Limpeza de Status**: Normalização textual e mapeamento para português com fallback `'Não Informado'`.
- **Conversão Multi-Formato de Data**: Teste de múltiplos formatos aceitos e extração do `ano_lancamento`.

In [3]:
DEFAULT_STATUS_FALLBACK = "Não Informado"

STATUS_TRANSLATION_MAP = {
    "RELEASED": "Lançado",
    "POST PRODUCTION": "Pós-Produção",
    "IN PRODUCTION": "Em Produção",
    "PLANNED": "Planejado",
    "RUMORED": "Rumores",
    "CANCELED": "Cancelado"
}

SUPPORTED_DATE_FORMATS = [
    "yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy",
    "MM/dd/yyyy", "dd-MM-yyyy", "MM-dd-yyyy"
]

TbInfoFilmesSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("titulo", StringType(), nullable=True),
    StructField("titulo_original", StringType(), nullable=True),
    StructField("data_lancamento", DateType(), nullable=True),
    StructField("ano_lancamento", IntegerType(), nullable=True),
    StructField("duracao_minutos", IntegerType(), nullable=True),
    StructField("idioma_original", StringType(), nullable=True),
    StructField("status_filme", StringType(), nullable=True),
    StructField("sinopse", StringType(), nullable=True),
    StructField("frase_divulgacao", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def clean_and_translate_status(column: Column) -> Column:
    # Limpeza e Tradução do Status: Antes de traduzir o status_filme
    # (Released -> Lançado; Post Production -> Pós-Produção; In Production -> Em Produção;
    # Planned -> Planejado; Rumored -> Rumores; Canceled -> Cancelado), você deve normalizar a coluna.
    # A coluna de status deve ser normalizada (removendo ruídos, hífens sobressalentes e padronizando a caixa)
    # antes da tradução dos termos para o português.
    normalized_status = regexp_replace(regexp_replace(upper(trim(column)), r"-", " "), r"\s+", " ")
    translation_mapping_expressions = [
        lit(item) for key_value_pair in STATUS_TRANSLATION_MAP.items() for item in key_value_pair
    ]
    status_lookup_map = create_map(translation_mapping_expressions)
    # Registros corrompidos ou não mapeáveis devem ser padronizados como 'Não Informado'.
    return coalesce(status_lookup_map[normalized_status], lit(DEFAULT_STATUS_FALLBACK))

def parse_multiformat_date(column: Column) -> Column:
    # Tratamento de Data Multi-Formato: Converta o campo de data de lançamento testando os diferentes
    # padrões presentes na origem de forma robusta. Apenas valores onde a conversão for estritamente
    # impossível devem ser tratados como ausentes (NULL).
    date_parsing_candidates = [try_to_date(column, single_date_format) for single_date_format in SUPPORTED_DATE_FORMATS]
    return coalesce(*date_parsing_candidates)

def transform_tb_movies_info(raw_movies_dataframe: DataFrame) -> DataFrame:
    # Deduplicação: A tabela deve conter unicidade por filme. Havendo registros duplicados na origem,
    # mantenha exclusivamente a versão mais recente com base na data de ingestão (ingestion_datetime).
    deduplicated = deduplicate_latest(raw_movies_dataframe, business_key_column="id")
    parsed_date_expression = parse_multiformat_date(col("release_date"))

    transformed_dataframe = deduplicated.select(
        col("id").cast("integer").alias("id_filme"),
        sanitize_string(col("title")).alias("titulo"),
        sanitize_string(col("original_title")).alias("titulo_original"),
        parsed_date_expression.alias("data_lancamento"),
        # Coluna Derivada: Criar a coluna ano_lancamento, extraída de data_lancamento.
        # Mapeamento de colunas em português e conversão de tipos conforme especificação do contrato.
        year(parsed_date_expression).alias("ano_lancamento"),
        col("runtime").cast("integer").alias("duracao_minutos"),
        sanitize_string(col("original_language")).alias("idioma_original"),
        clean_and_translate_status(col("status")).alias("status_filme"),
        sanitize_string(col("overview")).alias("sinopse"),
        sanitize_string(col("tagline")).alias("frase_divulgacao"),
        col("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbInfoFilmesSilverSchema)

dataframe_movies_info_bronze = read_lakehouse_table("bronze.tb_movies_info", BRONZE_DIR / "bronze.tb_movies_info")
dataframe_movies_info_silver = transform_tb_movies_info(dataframe_movies_info_bronze)
write_dataframe(dataframe_movies_info_silver, SILVER_DIR / "silver.tb_info_filmes", table_name="silver.tb_info_filmes")

display_dataframe(dataframe_movies_info_silver, 5)

execute_uniqueness_check("silver.tb_info_filmes", "unicidade_id_filme", dataframe_movies_info_silver, ["id_filme"])
execute_data_quality_check("silver.tb_info_filmes", "titulo_obrigatorio", dataframe_movies_info_silver, col("titulo").isNotNull())


DataFrame[id_filme: int, titulo: string, titulo_original: string, data_lancamento: date, ano_lancamento: int, duracao_minutos: int, idioma_original: string, status_filme: string, sinopse: string, frase_divulgacao: string, ingestion_datetime: timestamp]

[PASS] silver.tb_info_filmes | unicidade_id_filme: 0 duplicatas em 97879 registros.


[PASS] silver.tb_info_filmes | titulo_obrigatorio: 0/97879 falhas.


True

## 3. silver.tb_financeiro_filmes (Origem: bronze.tb_movies_financials)

### Regras de Negócio:
- **Higienização Monetária**: Extração de notações de escala (K, M, B), remoção de símbolos (`$`, `USD`) e conversão de valores $\le 0$ para `NULL`.
- **Conversão Cambial**: Conversão para BRL via taxa PTAX da `silver.tb_cotacao_dolar`.
- **Métricas de Lucro e Margem**: Cálculo protegido de lucro (USD e BRL) e margem percentual segura.

In [4]:
THOUSAND = 1_000
MILLION = 1_000_000
BILLION = 1_000_000_000
PERCENTAGE_FACTOR = 100
DECIMAL_PRECISION = "decimal(18,2)"

TbFinanceiroFilmesSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("orcamento_usd", DecimalType(18, 2), nullable=True),
    StructField("receita_usd", DecimalType(18, 2), nullable=True),
    StructField("orcamento_brl", DecimalType(18, 2), nullable=True),
    StructField("receita_brl", DecimalType(18, 2), nullable=True),
    StructField("lucro_usd", DecimalType(18, 2), nullable=True),
    StructField("lucro_brl", DecimalType(18, 2), nullable=True),
    StructField("margem_lucro_percentual", DecimalType(18, 2), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def sanitize_currency_column(column: Column) -> Column:
    # Tratar valores textuais que representam ausência de dado (ex.: 'Unknown', 'Não Informado') como NULL antes da conversão de tipo.
    uppercase_sentinels = [single_sentinel.upper() for single_sentinel in DEFAULT_SENTINEL_VALUES]
    raw_column_text = upper(trim(column))
    # Higienize as colunas de orçamento e receita para remover símbolos de moedas, pontuações de milhar e textos de ausência.
    cleaned_symbols = regexp_replace(regexp_replace(when(raw_column_text.isin(uppercase_sentinels), lit(None)).otherwise(raw_column_text), r"[$\s]|USD", ""), r",", "")

    # Tratamento de notações de escala abreviada (K para milhares, M para milhões, B para bilhões)
    extracted_thousands = regexp_extract(cleaned_symbols, r"^([0-9.]+)[Kk]$", 1)
    extracted_millions = regexp_extract(cleaned_symbols, r"^([0-9.]+)[Mm]$", 1)
    extracted_billions = regexp_extract(cleaned_symbols, r"^([0-9.]+)[Bb]$", 1)
    extracted_plain = regexp_extract(cleaned_symbols, r"^(-?[0-9.]+)$", 1)

    calculated_value = (
        when(extracted_thousands != "", extracted_thousands.cast("double") * THOUSAND)
        .when(extracted_millions != "", extracted_millions.cast("double") * MILLION)
        .when(extracted_billions != "", extracted_billions.cast("double") * BILLION)
        .when(extracted_plain != "", extracted_plain.cast("double"))
        .otherwise(lit(None))
    )
    # Converta as métricas para o tipo numérico decimal apropriado e garanta que valores zerados ou negativos sejam tratados como ausentes (NULL).
    return when(calculated_value > 0, calculated_value.cast(DECIMAL_PRECISION)).otherwise(lit(None))

def calculate_safe_percentage(
    numerator_column: Column,
    denominator_column: Column,
    scale_factor: int = PERCENTAGE_FACTOR,
    target_precision: str = DECIMAL_PRECISION
) -> Column:
    # Cálculo seguro de percentual evitando divisões por zero e propagando NULL para denominadores nulos ou inválidos
    valid_division =(
        numerator_column.isNotNull() 
        & denominator_column.isNotNull() 
        & (denominator_column > 0)
    )
    return when(valid_division, ((numerator_column / denominator_column) * scale_factor).cast(target_precision)).otherwise(lit(None))

def transform_tb_movies_financials(raw_financials_dataframe: DataFrame, exchange_rate_dollar_to_brl: float) -> DataFrame:
    # Deduplicação: Unicidade por filme mantendo o registro mais recente por data de ingestão
    deduplicated = deduplicate_latest(raw_financials_dataframe, business_key_column="id")
    rate_expr = lit(exchange_rate_dollar_to_brl).cast(DECIMAL_PRECISION)

    budget_usd_expr = sanitize_currency_column(col("budget"))
    revenue_usd_expr = sanitize_currency_column(col("revenue"))
    # Calcule os valores equivalentes em Reais (BRL) aplicando a taxa de cotação obtida.
    budget_brl_expr = (budget_usd_expr * rate_expr).cast(DECIMAL_PRECISION)
    revenue_brl_expr = (revenue_usd_expr * rate_expr).cast(DECIMAL_PRECISION)
    # Derive as colunas de Lucro (Dólar/Real) e Margem de Lucro Percentual, garantindo que operações aritméticas
    # com valores ausentes não invalidem o resultado e evitando divisões por zero.
    profit_usd_expr = (revenue_usd_expr - budget_usd_expr).cast(DECIMAL_PRECISION)
    profit_brl_expr = (revenue_brl_expr - budget_brl_expr).cast(DECIMAL_PRECISION)

    transformed_dataframe = deduplicated.select(
        col("id").cast("integer").alias("id_filme"),
        budget_usd_expr.alias("orcamento_usd"),
        revenue_usd_expr.alias("receita_usd"),
        budget_brl_expr.alias("orcamento_brl"),
        revenue_brl_expr.alias("receita_brl"),
        profit_usd_expr.alias("lucro_usd"),
        profit_brl_expr.alias("lucro_brl"),
        calculate_safe_percentage(profit_usd_expr, revenue_usd_expr).alias("margem_lucro_percentual"),
        col("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbFinanceiroFilmesSilverSchema)

# Leitura da cotação do dólar gerada na Silver para conversão cambial
dataframe_cotacao_silver = read_lakehouse_table("silver.tb_cotacao_dolar", SILVER_DIR / "silver.tb_cotacao_dolar")
latest_exchange_rate = float(dataframe_cotacao_silver.orderBy(col("data_cotacao").desc()).select("cotacao_compra").first()[0])
print(f"Taxa de Cotação utilizada (PTAX Compra - Silver): R$ {latest_exchange_rate:.4f}")

dataframe_financials_bronze = read_lakehouse_table("bronze.tb_movies_financials", BRONZE_DIR / "bronze.tb_movies_financials")
dataframe_financials_silver = transform_tb_movies_financials(dataframe_financials_bronze, latest_exchange_rate)
write_dataframe(dataframe_financials_silver, SILVER_DIR / "silver.tb_financeiro_filmes", table_name="silver.tb_financeiro_filmes")
dataframe_financials_silver.printSchema()

display_dataframe(dataframe_financials_silver, 10)

# Avaliação das métricas de integridade lendo a tabela já persistida na Silver,
# truncando a árvore de execução complexa de expressões para evitar o estouro de 64 KB no Janino
dataframe_financials_persisted = read_lakehouse_table("silver.tb_financeiro_filmes", SILVER_DIR / "silver.tb_financeiro_filmes")
execute_uniqueness_check("silver.tb_financeiro_filmes", "unicidade_id_filme", dataframe_financials_persisted, ["id_filme"])

condition = col("lucro_usd").isNull() | (col("lucro_usd") == (col("receita_usd") - col("orcamento_usd")))
execute_data_quality_check("silver.tb_financeiro_filmes", "lucro_consistente", dataframe_financials_persisted, condition)


Taxa de Cotação utilizada (PTAX Compra - Silver): R$ 5.1569


root
 |-- id_filme: integer (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(18,2) (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



DataFrame[id_filme: int, orcamento_usd: decimal(18,2), receita_usd: decimal(18,2), orcamento_brl: decimal(18,2), receita_brl: decimal(18,2), lucro_usd: decimal(18,2), lucro_brl: decimal(18,2), margem_lucro_percentual: decimal(18,2), ingestion_datetime: timestamp]

[PASS] silver.tb_financeiro_filmes | unicidade_id_filme: 0 duplicatas em 99006 registros.


26/09/19 23:53:12 ERROR CodeGenerator: Failed to compile the generated Java code.
org.codehaus.commons.compiler.InternalCompilerException: Compiling "GeneratedClass" in File 'generated.java', Line 1, Column 1: File 'generated.java', Line 44, Column 14: Compiling "hashAgg_doAggregateWithoutKey_0()"
	at org.codehaus.janino.UnitCompiler.compile2(UnitCompiler.java:402)
	at org.codehaus.janino.UnitCompiler.access$000(UnitCompiler.java:236)
	at org.codehaus.janino.UnitCompiler$2.visitCompilationUnit(UnitCompiler.java:363)
	at org.codehaus.janino.UnitCompiler$2.visitCompilationUnit(UnitCompiler.java:361)
	at org.codehaus.janino.Java$CompilationUnit.accept(Java.java:371)
	at org.codehaus.janino.UnitCompiler.compileUnit(UnitCompiler.java:361)
	at org.codehaus.janino.SimpleCompiler.cook(SimpleCompiler.java:264)
	at org.codehaus.janino.ClassBodyEvaluator.cook(ClassBodyEvaluator.java:294)
	at org.codehaus.janino.ClassBodyEvaluator.cook(ClassBodyEvaluator.java:288)
	at org.codehaus.janino.ClassBody

[PASS] silver.tb_financeiro_filmes | lucro_consistente: 0/99006 falhas.


True

## 4. silver.tb_metricas_engajamento (Origem: bronze.tb_movies_metrics)

### Regras de Negócio:
- **Higienização Numérica**: Substituição de vírgulas e casting seguro contra column shift.
- **Validação de Limites de Negócio**: Notas no intervalo [0, 10] e contagens $\ge 0$.

In [5]:
MINIMUM_RATING_VALUE = 0.0
MAXIMUM_RATING_VALUE = 10.0
MINIMUM_COUNT_VALUE = 0

TbMetricasEngajamentoSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("popularidade", DoubleType(), nullable=True),
    StructField("nota_media_tmdb", DoubleType(), nullable=True),
    StructField("qtd_votos_tmdb", IntegerType(), nullable=True),
    StructField("nota_media_imdb", DoubleType(), nullable=True),
    StructField("qtd_votos_imdb", IntegerType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def sanitize_numeric_metric(
    column: Column,
    target_data_type: str,
    minimum_value: float | None = None,
    maximum_value: float | None = None
) -> Column:
    # A coluna de popularidade apresenta inconsistências de pontuação e separadores decimais na origem.
    # Limpe a formatação numérica antes da conversão de tipo, garantindo que não sejam invalidados silenciosamente.
    # Devido ao Column Shift, aplique uma conversão de tipagem segura tratando textos incompatíveis como NULL.
    cleaned_numeric = regexp_replace(trim(column), ",", ".").cast(target_data_type)
    is_valid_metric = cleaned_numeric.isNotNull()
    # Identifique e trate inconsistências de escala e limites numéricos de negócio:
    # Notas médias fora da escala [0, 10] e contagens/popularidade negativas devem ser invalidadas e tratadas como NULL.
    if minimum_value is not None:
        is_valid_metric = is_valid_metric & (cleaned_numeric >= lit(minimum_value))
    if maximum_value is not None:
        is_valid_metric = is_valid_metric & (cleaned_numeric <= lit(maximum_value))
    return when(is_valid_metric, cleaned_numeric).otherwise(lit(None))

def transform_tb_movies_metrics(raw_metrics_dataframe: DataFrame) -> DataFrame:
    # Deduplicação: Unicidade por filme mantendo o registro mais recente por ingestion_datetime
    deduplicated = deduplicate_latest(raw_metrics_dataframe, business_key_column="id")
    transformed_dataframe = deduplicated.select(
        col("id").cast("integer").alias("id_filme"),
        sanitize_numeric_metric(col("popularity"), "double", minimum_value=0.0).alias("popularidade"),
        sanitize_numeric_metric(col("vote_average"), "double", minimum_value=MINIMUM_RATING_VALUE, maximum_value=MAXIMUM_RATING_VALUE).alias("nota_media_tmdb"),
        sanitize_numeric_metric(col("vote_count"), "integer", minimum_value=MINIMUM_COUNT_VALUE).alias("qtd_votos_tmdb"),
        sanitize_numeric_metric(col("averageRating"), "double", minimum_value=MINIMUM_RATING_VALUE, maximum_value=MAXIMUM_RATING_VALUE).alias("nota_media_imdb"),
        sanitize_numeric_metric(col("numVotes"), "integer", minimum_value=MINIMUM_COUNT_VALUE).alias("qtd_votos_imdb"),
        col("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbMetricasEngajamentoSilverSchema)

dataframe_metrics_bronze = read_lakehouse_table("bronze.tb_movies_metrics", BRONZE_DIR / "bronze.tb_movies_metrics")
dataframe_metrics_silver = transform_tb_movies_metrics(dataframe_metrics_bronze)
write_dataframe(dataframe_metrics_silver, SILVER_DIR / "silver.tb_metricas_engajamento", table_name="silver.tb_metricas_engajamento")
dataframe_metrics_silver.printSchema()
display_dataframe(dataframe_metrics_silver, 5)

execute_uniqueness_check("silver.tb_metricas_engajamento", "unicidade_id_filme", dataframe_metrics_silver, ["id_filme"])
execute_data_quality_check("silver.tb_metricas_engajamento", "faixa_nota_tmdb", dataframe_metrics_silver, col("nota_media_tmdb").isNull() | col("nota_media_tmdb").between(0.0, 10.0))


root
 |-- id_filme: integer (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



DataFrame[id_filme: int, popularidade: double, nota_media_tmdb: double, qtd_votos_tmdb: int, nota_media_imdb: double, qtd_votos_imdb: int, ingestion_datetime: timestamp]

[PASS] silver.tb_metricas_engajamento | unicidade_id_filme: 0 duplicatas em 99013 registros.


[PASS] silver.tb_metricas_engajamento | faixa_nota_tmdb: 0/99013 falhas.


True

## 5. silver.tb_avaliacoes_usuarios (Origem: bronze.tb_movies_reviews)

### Regras de Negócio:
- **Deduplicação Integral**: Unicidade estrita por combinação completa `(id, nome, nota, comentario)`.
- **Validação de Escala**: Notas no intervalo [0, 10].
- **Fallback de Comentários**: Preenchimento de ausentes/em branco com `'Sem comentário'`.

In [6]:
MINIMUM_USER_RATING = 0.0
MAXIMUM_USER_RATING = 10.0
DEFAULT_COMMENT_FALLBACK = "Sem comentário"

TbAvaliacoesUsuariosSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("nome_usuario", StringType(), nullable=True),
    StructField("nota_usuario", DoubleType(), nullable=True),
    StructField("comentario_usuario", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_tb_movies_reviews(raw_reviews_dataframe: DataFrame) -> DataFrame:
    # Remova registros integralmente duplicados (onde a combinação de filme, usuário, nota e comentário seja idêntica),
    # garantindo a unicidade das avaliações individuais.
    deduplicated = raw_reviews_dataframe.dropDuplicates(["id", "nome", "nota", "comentario"])
    trimmed_comment = trim(col("comentario"))
    valid_comment = trimmed_comment.isNotNull() & (trimmed_comment != "")
    parsed_rating = col("nota").cast("double")

    # Garanta que a nota atribuída pelo usuário respeite a escala permitida (0 a 10). Valores fora dessa faixa -> NULL.
    # Identifique comentários vazios ou compostos apenas por espaços e preencha com o texto padronizado 'Sem comentário'.
    transformed_dataframe = deduplicated.select(
        col("id").cast("integer").alias("id_filme"),
        trim(col("nome")).alias("nome_usuario"),
        when(parsed_rating.between(MINIMUM_USER_RATING, MAXIMUM_USER_RATING), parsed_rating).otherwise(lit(None)).alias("nota_usuario"),
        when(valid_comment, trimmed_comment).otherwise(lit(DEFAULT_COMMENT_FALLBACK)).alias("comentario_usuario"),
        col("ingestion_datetime")
    )
    return enforce_dataframe_schema(transformed_dataframe, TbAvaliacoesUsuariosSilverSchema)

dataframe_reviews_bronze = read_lakehouse_table("bronze.tb_movies_reviews", BRONZE_DIR / "bronze.tb_movies_reviews")
dataframe_reviews_silver = transform_tb_movies_reviews(dataframe_reviews_bronze)
write_dataframe(dataframe_reviews_silver, SILVER_DIR / "silver.tb_avaliacoes_usuarios", table_name="silver.tb_avaliacoes_usuarios")
dataframe_reviews_silver.printSchema()
display_dataframe(dataframe_reviews_silver, 5)

execute_data_quality_check("silver.tb_avaliacoes_usuarios", "faixa_nota_usuario", dataframe_reviews_silver, col("nota_usuario").isNull() | col("nota_usuario").between(0.0, 10.0))
execute_data_quality_check("silver.tb_avaliacoes_usuarios", "comentario_preenchido", dataframe_reviews_silver, col("comentario_usuario").isNotNull())


root
 |-- id_filme: integer (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: double (nullable = true)
 |-- comentario_usuario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



DataFrame[id_filme: int, nome_usuario: string, nota_usuario: double, comentario_usuario: string, ingestion_datetime: timestamp]

[PASS] silver.tb_avaliacoes_usuarios | faixa_nota_usuario: 0/32412 falhas.


[PASS] silver.tb_avaliacoes_usuarios | comentario_preenchido: 0/32412 falhas.


True

## 6. silver.tb_generos (Origem: bronze.tb_credits_and_tags)

### Regras de Negócio:
- **Explosão Atômica**: Desmembramento por separadores múltiplos (`,`, `;`, `|`).
- **Filtragem de Column Shift**: Retenção estrita dos gêneros canônicos cinematográficos.

In [7]:
KNOWN_GENRES_LIST = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music",
    "Mystery", "Romance", "Science Fiction", "TV Movie", "Thriller",
    "War", "Western"
]

TbGenerosSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("nome_genero", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def transform_tb_generos(raw_credits_dataframe: DataFrame) -> DataFrame:
    # Deduplicação: Unicidade por filme mantendo o registro mais recente por ingestion_datetime
    deduplicated = deduplicate_latest(raw_credits_dataframe, business_key_column="id")
    # Explodir (split + explode) a coluna genres, tratando a inconsistência de separadores (vírgula vs. ponto e vírgula vs. pipe) antes do split.
    normalized_genres = regexp_replace(col("genres"), r"[,;|]+", ",")
    sanitized_genre = trim(regexp_replace(col("_raw_genre"), r"""["'\[\]{}]""", ""))

    # Devido a falhas na estrutura de origem (Column Shift e separadores extras), remova resíduos em branco,
    # textos descritivos e valores numéricos deslocados que não pertençam ao domínio canônico de gêneros cinematográficos.
    transformed_dataframe = (
        deduplicated
        .filter(col("genres").isNotNull())
        .withColumn("_raw_genre", explode(split(normalized_genres, ",")))
        .withColumn("nome_genero", sanitized_genre)
        .filter(col("nome_genero").isin(KNOWN_GENRES_LIST))
        .select(
            col("id").cast("integer").alias("id_filme"),
            col("nome_genero"),
            col("ingestion_datetime")
        )
        .dropDuplicates(["id_filme", "nome_genero"])
    )
    return enforce_dataframe_schema(transformed_dataframe, TbGenerosSilverSchema)

dataframe_credits_bronze = read_lakehouse_table("bronze.tb_credits_and_tags", BRONZE_DIR / "bronze.tb_credits_and_tags")
dataframe_generos_silver = transform_tb_generos(dataframe_credits_bronze)
write_dataframe(dataframe_generos_silver, SILVER_DIR / "silver.tb_generos", table_name="silver.tb_generos")
dataframe_generos_silver.printSchema()
display_dataframe(dataframe_generos_silver, 10)

execute_data_quality_check("silver.tb_generos", "genero_canonico_preenchido", dataframe_generos_silver, col("nome_genero").isNotNull())


root
 |-- id_filme: integer (nullable = true)
 |-- nome_genero: string (nullable = false)
 |-- ingestion_datetime: timestamp (nullable = true)



DataFrame[id_filme: int, nome_genero: string, ingestion_datetime: timestamp]

[PASS] silver.tb_generos | genero_canonico_preenchido: 0/142149 falhas.


True

## 7. silver.tb_pessoas_empresas (Origem: bronze.tb_credits_and_tags)

### Regras de Negócio:
- **Dimensão Unificada**: Consolidação de `cast` (Ator), `directors` (Diretor), `writers` (Roteirista) e `production_companies` (Produtora).
- **Higienização e Padronização**: `initcap`, limpeza de caracteres especiais e eliminação de ruídos (URLs, imagens, números).

In [8]:
# Dimensão unificada que consolida quatro tipos de entidade em uma única tabela:
# cast -> 'Ator'; directors -> 'Diretor'; writers -> 'Roteirista'; production_companies -> 'Produtora'
ENTITY_TYPE_MAPPINGS = [
    ("cast", "Ator"),
    ("directors", "Diretor"),
    ("writers", "Roteirista"),
    ("production_companies", "Produtora")
]

TbPessoasEmpresasSilverSchema = StructType([
    StructField("id_filme", IntegerType(), nullable=True),
    StructField("nome_entidade", StringType(), nullable=True),
    StructField("tipo_entidade", StringType(), nullable=True),
    StructField("ingestion_datetime", TimestampType(), nullable=True),
])

def extract_and_clean_entities(dataframe: DataFrame, source_column_name: str, entity_type_label: str) -> DataFrame:
    # Desmembramento atômico por separadores múltiplos e padronização da capitalização (initcap)
    normalized_source = regexp_replace(col(source_column_name), r"[,;|]+", ",")
    sanitized_name = initcap(trim(regexp_replace(col("_raw_entity"), r"""["'\[\]{}]""", "")))

    # Expurgar resíduos de Column Shift: URLs, arquivos de imagem, valores puramente numéricos e sentinelas
    return (
        dataframe
        .filter(col(source_column_name).isNotNull())
        .withColumn("_raw_entity", explode(split(normalized_source, ",")))
        .withColumn("nome_entidade", sanitized_name)
        .withColumn("tipo_entidade", lit(entity_type_label))
        .filter(
            col("nome_entidade").isNotNull()
            & (col("nome_entidade") != "")
            & (~upper(col("nome_entidade")).isin(DEFAULT_SENTINEL_VALUES))
            & (~col("nome_entidade").rlike("^[0-9. -]+$"))
            & (~col("nome_entidade").rlike(r"(?i)\.(jpg|png|jpeg|webp)"))
            & (~col("nome_entidade").rlike(r"(?i)^(http|https|www\.)"))
            & (length(col("nome_entidade")) > 1)
        )
        .select(
            col("id").cast("integer").alias("id_filme"),
            col("nome_entidade"),
            col("tipo_entidade"),
            col("ingestion_datetime")
        )
    )

def transform_tb_pessoas_empresas(raw_credits_dataframe: DataFrame) -> DataFrame:
    # Deduplicação: Unicidade por filme mantendo o registro mais recente por ingestion_datetime
    deduplicated = deduplicate_latest(raw_credits_dataframe, business_key_column="id")
    # Consolidação dos quatro fluxos em modelo unificado categorizado pelo tipo de atuação
    extracted_dfs = [
        extract_and_clean_entities(deduplicated, source_col, label)
        for source_col, label in ENTITY_TYPE_MAPPINGS
    ]
    unified_df = extracted_dfs[0]
    for additional_df in extracted_dfs[1:]:
        unified_df = unified_df.unionByName(additional_df)

    # Elimine registros duplicados pela tríade (id_filme, nome_entidade, tipo_entidade)
    transformed_dataframe = unified_df.dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
    return enforce_dataframe_schema(transformed_dataframe, TbPessoasEmpresasSilverSchema)

dataframe_pessoas_empresas_silver = transform_tb_pessoas_empresas(dataframe_credits_bronze)
write_dataframe(dataframe_pessoas_empresas_silver, SILVER_DIR / "silver.tb_pessoas_empresas", table_name="silver.tb_pessoas_empresas")
dataframe_pessoas_empresas_silver.printSchema()
display_dataframe(dataframe_pessoas_empresas_silver, 10)

execute_data_quality_check("silver.tb_pessoas_empresas", "nome_entidade_obrigatorio", dataframe_pessoas_empresas_silver, col("nome_entidade").isNotNull())


root
 |-- id_filme: integer (nullable = true)
 |-- nome_entidade: string (nullable = false)
 |-- tipo_entidade: string (nullable = false)
 |-- ingestion_datetime: timestamp (nullable = true)



DataFrame[id_filme: int, nome_entidade: string, tipo_entidade: string, ingestion_datetime: timestamp]

[PASS] silver.tb_pessoas_empresas | nome_entidade_obrigatorio: 0/889993 falhas.


True

## 8. Consolidação e Persistência do Log de Data Quality (Auditoria)

Persistência e visualização do histórico consolidado de auditoria de qualidade na camada Silver.

In [9]:
persist_data_quality_log(SILVER_DIR)
dataframe_dq_log = read_lakehouse_table("silver.tb_data_quality_log", SILVER_DIR / "tb_data_quality_log")
display_dataframe(dataframe_dq_log.orderBy(col("checked_at").desc()), 20)


Log de Data Quality persistido com sucesso em: /home/miguelsb/workspace/visagio/data/silver/tb_data_quality_log


DataFrame[table_name: string, check_name: string, total_records: bigint, failed_records: bigint, passed: boolean, checked_at: timestamp]